# SE-801 Lecture 5 Nonlinear activations and multilayer perceptrons

**Central question:** What can an affine classifier never represent, regardless of training time?

This self-contained notebook accompanies the lecture slides and student study guide. Predict and calculate before revealing each output. It uses synthetic data and runs offline on CPU with PyTorch, NumPy and Matplotlib. It embeds every implementation cell and does not import a local helper file or download data.

**Class route:** 80 teaching minutes, 20-minute break, 70 teaching minutes, then 10 minutes of flexibility. RUN 19–20 are reserve experiments. Test labels first enter performance reporting at RUN 21.

**Notation:** scratch batches use row-wise examples and weights `[input, output]`; `nn.Linear` stores the transposed weight. Classification uses raw logits, integer targets and mean cross-entropy. Local derivative discussion prepares later backpropagation work without adding Lecture 6's detailed initialization or optimizer topics.

**Output folder:** by default `lecture_05_outputs` under the notebook's working directory. A local build can set `ANN_L5_OUTPUT_DIR`. Installing the three dependencies is needed only if your environment lacks them.

**Sources:** [D2L MLP concepts](https://d2l.ai/chapter_multilayer-perceptrons/mlp.html), [D2L implementation](https://d2l.ai/chapter_multilayer-perceptrons/mlp-implementation.html), [PyTorch Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html), [PyTorch CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html). All experiment values arise from the code below.


## RUN 0 Setup

**Predict:** What changes when a model predicts a category rather than a measured value?

In [ ]:
from pathlib import Path
import os
import json
import math
import copy
import csv
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
torch.set_default_dtype(torch.float64)
np.random.seed(51)
torch.manual_seed(51)
OUT = Path(os.environ.get('ANN_L5_OUTPUT_DIR', 'lecture_05_outputs'))
OUT.mkdir(parents=True, exist_ok=True)
FIG = OUT / 'figures'
FIG.mkdir(exist_ok=True)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .18})
BLUE, GOLD, TEAL = '#246B8E', '#B87521', '#218477'
evidence = {'versions': {'torch': torch.__version__, 'numpy': np.__version__},
            'device': 'cpu', 'dtype': 'float64'}
chart_data = {}

def show_save(fig, name):
    fig.tight_layout()
    fig.savefig(FIG / (name + '.png'), dpi=160, bbox_inches='tight')
    plt.show()
    plt.close(fig)

def class_scatter(ax, X, y, title):
    a = np.asarray(X); labels = np.asarray(y)
    for k, color, marker in [(0, BLUE, 'o'), (1, GOLD, '^')]:
        points = a[labels == k]
        ax.scatter(points[:, 0], points[:, 1], c=color, marker=marker,
                   s=30, alpha=.75, label='class ' + str(k))
    ax.set(xlabel='feature 1', ylabel='feature 2', title=title)
    ax.legend()

print('CPU | PyTorch', torch.__version__, '| NumPy', np.__version__)

## RUN 1 AND OR and XOR

**Predict:** Draw the AND, OR and XOR classes before seeing the plots. Which admit a line?

In [ ]:
corners = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
and_y = torch.tensor([0, 0, 0, 1])
or_y = torch.tensor([0, 1, 1, 1])
xor_y = torch.tensor([0, 1, 1, 0])
assert torch.equal((corners.sum(1) - 1.5 >= 0).long(), and_y)
assert torch.equal((corners.sum(1) - .5 >= 0).long(), or_y)
print('input | AND OR XOR')
for x, a, o, q in zip(corners.tolist(), and_y, or_y, xor_y):
    print(x, '|', a.item(), o.item(), q.item())
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
for ax, y, title in zip(axes, [and_y, or_y, xor_y], ['AND', 'OR', 'XOR']):
    class_scatter(ax, corners, y, title)
    ax.set(xlim=(-.3, 1.3), ylim=(-.3, 1.3))
axes[0].plot([.2, 1.2], [1.3, .3], color=TEAL, label='boundary')
axes[1].plot([-.2, .7], [.7, -.2], color=TEAL)
show_save(fig, '01_logic_geometry')
chart_data['corners'] = {'X': corners.tolist(), 'AND': and_y.tolist(),
                         'OR': or_y.tolist(), 'XOR': xor_y.tolist()}

## RUN 2 Perceptron learning and its limit

**Predict:** Can a repeated weight-update rule change the family of decision boundaries?

In [ ]:
def perceptron(X, y, epochs=25, eta=.2):
    w = torch.zeros(X.shape[1]); b = torch.tensor(0.)
    mistakes = []
    for _ in range(epochs):
        count = 0
        for xi, yi in zip(X, y):
            pred = int((xi @ w + b).item() >= 0)
            error = int(yi) - pred
            if error:
                w += eta * error * xi
                b += eta * error
                count += 1
        mistakes.append(count)
    return w, b, mistakes

for label, target in [('AND', and_y), ('OR', or_y), ('XOR', xor_y)]:
    pw, pb, mistakes = perceptron(corners, target)
    print(label, 'last five mistake counts:', mistakes[-5:])
    if label != 'XOR':
        assert mistakes[-1] == 0
    else:
        assert not torch.equal((corners @ pw + pb >= 0).long(), target)
print('The geometric contradiction establishes the XOR limit; this run illustrates it.')

## RUN 3 Two affine layers collapse

**Predict:** Expand both matrix products on paper. Which effective weight and bias remain?

In [ ]:
g = torch.Generator().manual_seed(52)
X_demo = torch.randn(5, 2, generator=g)
W1 = torch.randn(2, 3, generator=g); b1 = torch.randn(3, generator=g)
W2 = torch.randn(3, 2, generator=g); b2 = torch.randn(2, generator=g)
two_layers = (X_demo @ W1 + b1) @ W2 + b2
collapsed = X_demo @ (W1 @ W2) + (b1 @ W2 + b2)
collapse_error = (two_layers - collapsed).abs().max().item()
assert torch.allclose(two_layers, collapsed, atol=1e-12, rtol=1e-12)
print('Maximum affine-collapse difference:', collapse_error)
evidence['collapse_max_error'] = collapse_error

## RUN 4 Activation functions

**Predict:** Predict the ranges and value at zero of each activation.

In [ ]:
z = torch.linspace(-5, 5, 101)
activation_values = {'sigmoid': torch.sigmoid(z), 'tanh': torch.tanh(z),
                     'ReLU': torch.relu(z)}
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
for ax, (name, values), color in zip(axes, activation_values.items(), [BLUE, TEAL, GOLD]):
    ax.plot(z, values, color=color, lw=2.5)
    ax.set(title=name, xlabel='preactivation z', ylabel='activation')
show_save(fig, '02_activations')
chart_data['activations'] = {'z': z.tolist(), **{k: v.tolist() for k,v in activation_values.items()}}

## RUN 5 Local derivatives and saturation

**Predict:** Which inputs transmit a local gradient? What happens at a saturated sigmoid?

In [ ]:
derivatives = {'sigmoid': torch.sigmoid(z) * (1 - torch.sigmoid(z)),
               'tanh': 1 - torch.tanh(z).square(), 'ReLU': (z > 0).to(z.dtype)}
fig, ax = plt.subplots(figsize=(8, 3.8))
for (name, values), color in zip(derivatives.items(), [BLUE, TEAL, GOLD]):
    ax.plot(z, values, label=name, color=color, lw=2)
ax.set(xlabel='preactivation z', ylabel='local derivative', title='Local derivatives')
ax.legend(); show_save(fig, '03_derivatives')
trial_z = torch.tensor([-3., 0., 2.], requires_grad=True)
torch.relu(trial_z).sum().backward()
assert torch.equal(trial_z.grad, torch.tensor([0., 0., 1.]))
print('ReLU([-3,0,2]):', torch.relu(trial_z).tolist(), '| autograd:', trial_z.grad.tolist())
print('sigmoid derivative at 0:', .25, '| at 5:', derivatives['sigmoid'][-1].item())
chart_data['derivatives'] = {'z': z.tolist(), **{k: v.tolist() for k,v in derivatives.items()}}

## RUN 6 Hand forward calculation

**Predict:** Calculate the two preactivations, apply ReLU, then calculate the output score.

In [ ]:
x_hand = torch.tensor([1., 2.])
W_hand = torch.tensor([[1., -1.], [1., 1.]])
b_hand = torch.tensor([0., -1.])
z_hand = W_hand @ x_hand + b_hand
h_hand = torch.relu(z_hand)
score_hand = torch.tensor([2., -1.]) @ h_hand + .5
assert torch.equal(z_hand, torch.tensor([-1., 2.]))
assert torch.equal(h_hand, torch.tensor([0., 2.])) and score_hand.item() == -1.5
print('preactivation:', z_hand.tolist(), 'hidden:', h_hand.tolist(), 'score:', score_hand.item())
evidence['hand_forward'] = {'z': z_hand.tolist(), 'h': h_hand.tolist(), 'score': score_hand.item()}

# Break

20 minutes. Resume with the constructed XOR representation.

## RUN 7 Construct a representation of XOR

**Predict:** Calculate hidden features for all four XOR inputs before running the construction.

In [ ]:
H_exact = torch.stack([torch.relu(corners[:,0] - corners[:,1]),
                       torch.relu(corners[:,1] - corners[:,0])], dim=1)
exact_score = H_exact.sum(1)
exact_pred = (exact_score >= .5).long()
assert torch.equal(exact_pred, xor_y)
print('hidden features:', H_exact.tolist())
print('scores:', exact_score.tolist(), '| predictions:', exact_pred.tolist())
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
class_scatter(axes[0], corners, xor_y, 'Original XOR coordinates')
class_scatter(axes[1], H_exact, xor_y, 'Constructed hidden coordinates')
axes[1].set(xlabel='h1 = ReLU(x1-x2)', ylabel='h2 = ReLU(x2-x1)')
axes[1].plot([-.2,.7],[.7,-.2],color=TEAL)
show_save(fig, '04_xor_representation')
chart_data['exact_hidden'] = {'X': H_exact.tolist(), 'y': xor_y.tolist()}

## RUN 8 Batch shapes and parameters

**Predict:** Count weights and biases for a 2-input, 3-hidden, 2-output network. Does batch size matter?

In [ ]:
batch = torch.zeros(8, 2)
shape_model = nn.Sequential(nn.Linear(2, 3), nn.ReLU(), nn.Linear(3, 2))
assert shape_model(batch).shape == (8, 2)
shape_count = sum(p.numel() for p in shape_model.parameters())
assert shape_count == 17
for name, param in shape_model.named_parameters():
    print(name, list(param.shape), 'parameters:', param.numel())
print('Batch input:', list(batch.shape), '| logits:', list(shape_model(batch).shape), '| total:', shape_count)

## RUN 9 Scratch and framework equivalence

**Predict:** Why must scratch weight matrices be transposed when copying them into nn.Linear?

In [ ]:
def scratch_parameters(seed=31, width=8):
    gen = torch.Generator().manual_seed(seed)
    return [(torch.randn(2, width, generator=gen) * .4).requires_grad_(),
            torch.zeros(width, requires_grad=True),
            (torch.randn(width, 2, generator=gen) * .4).requires_grad_(),
            torch.zeros(2, requires_grad=True)]

def scratch_logits(X, params, activation=True):
    W1, b1, W2, b2 = params
    preactivation = X @ W1 + b1
    hidden = torch.relu(preactivation) if activation else preactivation
    return hidden @ W2 + b2

def matching_module(params, activation=True):
    W1, b1, W2, b2 = params
    model = nn.Sequential(nn.Linear(W1.shape[0], W1.shape[1]),
                          nn.ReLU() if activation else nn.Identity(),
                          nn.Linear(W2.shape[0], W2.shape[1]))
    with torch.no_grad():
        model[0].weight.copy_(W1.T); model[0].bias.copy_(b1)
        model[2].weight.copy_(W2.T); model[2].bias.copy_(b2)
    return model

params_check = scratch_parameters(seed=19, width=3)
module_check = matching_module(params_check)
scratch_out = scratch_logits(corners, params_check)
module_out = module_check(corners)
criterion = nn.CrossEntropyLoss()
loss_s = criterion(scratch_out, xor_y); loss_m = criterion(module_out, xor_y)
loss_s.backward(); loss_m.backward()
gradient_pairs = [(params_check[0].grad, module_check[0].weight.grad.T),
                  (params_check[1].grad, module_check[0].bias.grad),
                  (params_check[2].grad, module_check[2].weight.grad.T),
                  (params_check[3].grad, module_check[2].bias.grad)]
assert torch.allclose(scratch_out, module_out, atol=1e-12)
assert torch.allclose(loss_s, loss_m, atol=1e-12)
for a,b in gradient_pairs: assert torch.allclose(a,b,atol=1e-12)
print('Matching logits, loss and all gradients:', loss_s.item())

## RUN 10 One identical SGD step

**Predict:** If the gradients, parameters and learning rate agree, should one SGD update agree?

In [ ]:
eta_check = .1
with torch.no_grad():
    for param in params_check: param -= eta_check * param.grad
optimizer_check = torch.optim.SGD(module_check.parameters(), lr=eta_check)
optimizer_check.step()
after_error = (scratch_logits(corners, params_check) - module_check(corners)).abs().max().item()
assert after_error < 1e-12
print('After matched SGD update, maximum logit difference:', after_error)
evidence['equivalence'] = {'initial_loss': loss_s.item(), 'after_update_max_error': after_error,
                          'forward_loss_gradient_agreement': True}

## RUN 11 Independent noisy XOR splits

**Predict:** Which split is allowed to fit the feature mean and scale? What population do these draws represent?

In [ ]:
def make_split(seed, per_corner):
    rng = np.random.default_rng(seed)
    centers = np.array([[-1.,-1.],[-1.,1.],[1.,-1.],[1.,1.]])
    X = np.concatenate([rng.normal(center, .35, size=(per_corner,2)) for center in centers])
    y = np.repeat([0,1,1,0], per_corner)
    order = rng.permutation(len(y))
    return torch.tensor(X[order]), torch.tensor(y[order], dtype=torch.long)

X_train_raw, y_train = make_split(501, 60)
X_val_raw, y_val = make_split(502, 20)
X_test_raw, y_test = make_split(503, 20)
mu = X_train_raw.mean(0); scale = X_train_raw.std(0, unbiased=False)
X_train = (X_train_raw-mu)/scale; X_val = (X_val_raw-mu)/scale
X_test = (X_test_raw-mu)/scale
assert X_train.shape == (240,2) and X_val.shape == X_test.shape == (80,2)
print('Train/validation/test:', len(y_train), len(y_val), len(y_test))
print('Training-only center and scale:', mu.tolist(), scale.tolist())
print('Test labels generated but not used for tuning or reporting so far.')
fig, ax = plt.subplots(figsize=(6,4.5)); class_scatter(ax,X_train_raw,y_train,'Synthetic training split')
show_save(fig,'05_training_data')
chart_data['train_data']={'X':X_train_raw.tolist(),'y':y_train.tolist()}
for label,X,y in [('train',X_train_raw,y_train),('validation',X_val_raw,y_val),('test',X_test_raw,y_test)]:
    with (OUT/(label+'_synthetic.csv')).open('w',newline='') as handle:
        writer=csv.writer(handle); writer.writerow(['feature_1','feature_2','class'])
        writer.writerows([*xi,int(yi)] for xi,yi in zip(X.tolist(),y.tolist()))
evidence['protocol']={'split_seeds':[501,502,503],'train_n':240,'validation_n':80,'test_n':80,
    'noise_sd':.35,'hidden_width':8,'initialization_seed':31,'initial_weight_sd':.4,
    'epochs':120,'batch_size':32,'learning_rate':.1,'batch_order_seed':900,
    'selection':'lowest validation cross-entropy; earliest epoch wins ties',
    'claim':'independent draws from this fixed synthetic mixture only',
    'train_mean':mu.tolist(),'train_scale':scale.tolist()}

## RUN 12 Shared training and evaluation helpers

**Predict:** Read the scratch loop. Locate clearing, forward/loss, backward, update, validation and checkpoint selection.

In [ ]:
@torch.no_grad()
def metrics(predict, X, y):
    logits = predict(X); pred = logits.argmax(1)
    matrix = torch.zeros(2,2,dtype=torch.long)
    for truth,guess in zip(y,pred): matrix[truth,guess] += 1
    recalls = matrix.diag().to(torch.float64)/matrix.sum(1).clamp_min(1)
    return {'ce':criterion(logits,y).item(),'accuracy':(pred==y).double().mean().item(),
            'balanced_accuracy':recalls.mean().item(),'recall':recalls.tolist(),
            'confusion':matrix.tolist()}

def train_scratch(seed=31, activation=True):
    params = scratch_parameters(seed)
    rng = torch.Generator().manual_seed(900)
    history=[]; best_loss=float('inf'); checkpoint=None; best_epoch=None
    for epoch in range(1,121):
        order=torch.randperm(len(y_train),generator=rng)
        for idx in order.split(32):
            for p in params: p.grad=None
            loss=criterion(scratch_logits(X_train[idx],params,activation),y_train[idx])
            loss.backward()
            with torch.no_grad():
                for p in params: p -= .1*p.grad
        predict=lambda X:scratch_logits(X,params,activation)
        train=metrics(predict,X_train,y_train); val=metrics(predict,X_val,y_val)
        history.append({'epoch':epoch,'train_ce':train['ce'],'val_ce':val['ce'],
                        'train_accuracy':train['accuracy'],'val_accuracy':val['accuracy']})
        if val['ce'] < best_loss:
            best_loss=val['ce']; best_epoch=epoch
            checkpoint=[p.detach().clone() for p in params]
    return {'params':checkpoint,'history':history,'best_epoch':best_epoch,'activation':activation,
            'parameter_count':sum(p.numel() for p in params)}

def train_framework(seed=31, kind='mlp'):
    if kind=='linear':
        torch.manual_seed(seed)
        model=nn.Linear(2,2)
        with torch.no_grad():
            model.weight.normal_(0,.4); model.bias.zero_()
    else:
        model=matching_module(scratch_parameters(seed),activation=(kind=='mlp'))
    opt=torch.optim.SGD(model.parameters(),lr=.1)
    rng=torch.Generator().manual_seed(900)
    history=[]; best_loss=float('inf'); checkpoint=None; best_epoch=None
    for epoch in range(1,121):
        model.train(); order=torch.randperm(len(y_train),generator=rng)
        for idx in order.split(32):
            opt.zero_grad(); loss=criterion(model(X_train[idx]),y_train[idx])
            loss.backward(); opt.step()
        model.eval(); train=metrics(model,X_train,y_train); val=metrics(model,X_val,y_val)
        history.append({'epoch':epoch,'train_ce':train['ce'],'val_ce':val['ce'],
                        'train_accuracy':train['accuracy'],'val_accuracy':val['accuracy']})
        if val['ce'] < best_loss:
            best_loss=val['ce']; best_epoch=epoch
            checkpoint=copy.deepcopy(model.state_dict())
    model.load_state_dict(checkpoint); model.eval()
    return {'model':model,'history':history,'best_epoch':best_epoch,
            'parameter_count':sum(p.numel() for p in model.parameters())}

print('Same splits, loss, batch order, learning rate and maximum epoch budget.')
print('MLP and affine control also share initial tensors and parameter count.')

## RUN 13 Linear baseline

**Predict:** What result would a linear model establish on an XOR-like population?

In [ ]:
linear_run=train_framework(kind='linear')
print('Linear validation:',metrics(linear_run['model'],X_val,y_val),
      '| selected epoch:',linear_run['best_epoch'])

## RUN 14 Equal-parameter affine control

**Predict:** This model has the same parameter count as the MLP. Can depth without activation express XOR?

In [ ]:
affine_run=train_framework(kind='affine')
print('Affine-stack validation:',metrics(affine_run['model'],X_val,y_val),
      '| parameters:',affine_run['parameter_count'])
assert affine_run['parameter_count']==42

## RUN 15 Train the MLP in two implementations

**Predict:** What checks distinguish a correct second implementation from a lucky similar accuracy?

In [ ]:
scratch_run=train_scratch()
mlp_run=train_framework(kind='mlp')
scratch_predict=lambda X:scratch_logits(X,scratch_run['params'])
framework_predict=mlp_run['model']
train_agreement=(scratch_predict(X_train)-framework_predict(X_train)).abs().max().item()
assert train_agreement < 1e-10
assert scratch_run['best_epoch']==mlp_run['best_epoch']
assert mlp_run['parameter_count']==42
print('MLP validation:',metrics(framework_predict,X_val,y_val))
print('Selected epoch:',mlp_run['best_epoch'],'| scratch/framework maximum logit difference:',train_agreement)
evidence['equivalence']['trained_max_logit_error']=train_agreement

## RUN 16 Learning curves

**Predict:** Compare the same metric across models and splits. Where does extra capacity help?

In [ ]:
runs={'linear':linear_run,'affine stack':affine_run,'ReLU MLP':mlp_run}
fig,axes=plt.subplots(1,2,figsize=(11,4))
for (name,result),color in zip(runs.items(),[BLUE,GOLD,TEAL]):
    history=result['history']; epochs=[r['epoch'] for r in history]
    axes[0].plot(epochs,[r['train_ce'] for r in history],label=name,color=color)
    axes[1].plot(epochs,[r['val_ce'] for r in history],label=name,color=color)
for ax,title in zip(axes,['Training cross-entropy','Validation cross-entropy']):
    ax.set(xlabel='epoch',ylabel='mean cross-entropy',title=title); ax.legend()
show_save(fig,'06_learning_curves')
chart_data['curves']={name:result['history'] for name,result in runs.items()}

## RUN 17 Validation report and majority baseline

**Predict:** What does a balanced-sample majority baseline achieve? Which evidence selects the checkpoint?

In [ ]:
validation_report={}
for name,result in runs.items():
    report=metrics(result['model'],X_val,y_val)
    report.update({'best_epoch':result['best_epoch'],'parameter_count':result['parameter_count']})
    validation_report[name]=report
    print(name,report)
majority_label=int(torch.bincount(y_train).argmax())
majority_accuracy=(y_val==majority_label).double().mean().item()
assert majority_accuracy==.5
print('Fixed training-majority baseline (tie selects class 0):',majority_accuracy)
evidence['validation']=validation_report; evidence['majority_validation_accuracy']=majority_accuracy

## RUN 18 Decision regions and a hidden projection

**Predict:** Does the MLP boundary follow one line? Does a projection onto two hidden units preserve all information?

In [ ]:
grid_axis=np.linspace(-2,2,61)
gx,gy=np.meshgrid(grid_axis,grid_axis)
grid_raw=torch.tensor(np.column_stack([gx.ravel(),gy.ravel()]))
grid_scaled=(grid_raw-mu)/scale
fig=plt.figure(figsize=(12,4.5))
layout=fig.add_gridspec(2,3,height_ratios=[1,.08])
axes=[fig.add_subplot(layout[0,i]) for i in range(3)]
probability_cmap=LinearSegmentedColormap.from_list('class_probability',[BLUE,'#F4F4F4',GOLD])
for ax,(name,result) in zip(axes,runs.items()):
    with torch.no_grad(): p1=torch.softmax(result['model'](grid_scaled),1)[:,1].numpy()
    probability_plot=ax.contourf(gx,gy,p1.reshape(gx.shape),levels=np.linspace(0,1,11),cmap=probability_cmap,alpha=.65)
    ax.contour(gx,gy,p1.reshape(gx.shape),levels=[.5],colors='black',linewidths=1)
    class_scatter(ax,X_val_raw,y_val,name); ax.set(xlim=(-2,2),ylim=(-2,2))
colour_axis=fig.add_subplot(layout[1,:])
fig.colorbar(probability_plot,cax=colour_axis,orientation='horizontal',label='Predicted probability of class 1 (background). Black line: p = 0.5')
show_save(fig,'07_decision_regions')
with torch.no_grad():
    hidden_val=torch.relu(mlp_run['model'][0](X_val))
fig,ax=plt.subplots(figsize=(6,4)); class_scatter(ax,hidden_val[:,:2],y_val,'Two of eight learned hidden units')
ax.set(xlabel='hidden unit 1',ylabel='hidden unit 2'); show_save(fig,'08_hidden_projection')
print('A two-unit projection can overlap even when the full eight-unit representation classifies well.')
small_axis=np.linspace(-2,2,25); sx,sy=np.meshgrid(small_axis,small_axis)
small_raw=torch.tensor(np.column_stack([sx.ravel(),sy.ravel()]))
with torch.no_grad(): small_pred=mlp_run['model']((small_raw-mu)/scale).argmax(1)
chart_data['mlp_region']={'X':small_raw.tolist(),'y':small_pred.tolist()}
chart_data['validation_data']={'X':X_val_raw.tolist(),'y':y_val.tolist()}

## RUN 19 Reserve experiment repeated initializations

**Predict:** RESERVE: should we report every registered seed or select only the best one?

In [ ]:
seed_reports=[]
for seed in [31,32,33,34,35]:
    result=mlp_run if seed==31 else train_framework(seed=seed,kind='mlp')
    report=metrics(result['model'],X_val,y_val)
    seed_reports.append({'seed':seed,'best_epoch':result['best_epoch'],
                         'val_ce':report['ce'],'val_accuracy':report['accuracy']})
print('All preregistered initialization seeds:',seed_reports)
print('These repeats share one validation sample; they do not estimate population uncertainty.')
evidence['seed_repeats']=seed_reports

## RUN 20 Reserve experiment an inactive ReLU

**Predict:** RESERVE: what local gradient reaches a ReLU whose preactivation remains negative?

In [ ]:
dead_z=torch.tensor([-2.,-1.],requires_grad=True)
torch.relu(dead_z).sum().backward()
assert torch.equal(dead_z.grad,torch.zeros(2))
print('Inactive local ReLU derivatives:',dead_z.grad.tolist())
print('A finite-difference check should avoid zero, where ReLU is nondifferentiable.')

## RUN 21 Frozen held-out test evaluation

**Predict:** State everything frozen before exposing test results. Can those results certify new-subject transfer?

In [ ]:
print('Before evaluation: model family, preprocessing, width, seed, loss, checkpoint and decision rule are frozen.')
test_report={name:metrics(result['model'],X_test,y_test) for name,result in runs.items()}
for name,report in test_report.items(): print(name,report)
assert test_report['ReLU MLP']['accuracy']>.85
assert validation_report['ReLU MLP']['ce'] < validation_report['linear']['ce']
assert validation_report['ReLU MLP']['ce'] < validation_report['affine stack']['ce']
evidence['test']=test_report
evidence['seed_repeats_used_to_change_primary_seed']=False
evidence['test_used_for_selection']=False

## RUN 22 Export evidence and finish

**Predict:** What evidence belongs beside a performance claim? Name one limitation of this synthetic experiment.

In [ ]:
with (OUT/'experiment_results.json').open('w') as handle: json.dump(evidence,handle,indent=2)
with (OUT/'chart_data.json').open('w') as handle: json.dump(chart_data,handle)
with (OUT/'learning_curves.csv').open('w',newline='') as handle:
    writer=csv.DictWriter(handle,fieldnames=['model','epoch','train_ce','val_ce','train_accuracy','val_accuracy'])
    writer.writeheader()
    for name,result in runs.items():
        for row in result['history']: writer.writerow({'model':name,**row})
print('Evidence saved to',OUT.resolve())
print('Exit: identify representation, optimization and generalization issues separately.')